
# 04b_filter_mintaka_for_reconstruction

Фильтрация **Mintaka multihop**-кандидатов для твоего benchmark-а.

Что делает ноутбук:
- читает результаты из `artifacts_mintaka_stage1`;
- отфильтровывает **shallow bridge-questions** и прочие слишком лёгкие случаи;
- повышает вес вопросов с **несколькими gold-ответами**, `list_like`, temporal/comparison/intersection сигналами;
- убирает однотипные шаблоны по **surface template**;
- собирает shortlist для ручного просмотра.

Основная идея:
- **Mintaka** остаётся источником сложных паттернов;
- но в benchmark не стоит тащить все multihop-вопросы подряд;
- сначала надо сжать их до более сильного и разнообразного слоя.


In [1]:

# %pip install -q pandas


In [2]:

from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd


In [3]:

# =========================
# КОНФИГ
# =========================

PROJECT_ROOT = Path(".").resolve()

INPUT_STAGE1_DIR = PROJECT_ROOT / "artifacts_mintaka_stage1"
INPUT_JSONL_DIR = INPUT_STAGE1_DIR / "jsonl"
INPUT_CSV_DIR = INPUT_STAGE1_DIR / "csv"

# Предпочитаем более полный multihop-файл, если он есть
CANDIDATE_PATHS = [
    INPUT_JSONL_DIR / "mintaka_multihop_candidates.jsonl",
    INPUT_JSONL_DIR / "mintaka_all_normalized.jsonl",
    INPUT_CSV_DIR / "mintaka_multihop_translation_minimal.csv",
]

OUTPUT_DIR = PROJECT_ROOT / "artifacts_mintaka_stage2"
OUT_JSONL_DIR = OUTPUT_DIR / "jsonl"
OUT_CSV_DIR = OUTPUT_DIR / "csv"
OUT_REPORTS_DIR = OUTPUT_DIR / "reports"

for p in [OUTPUT_DIR, OUT_JSONL_DIR, OUT_CSV_DIR, OUT_REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Пороговые и балансировочные параметры
KEEP_MIN_SCORE = 12
PATTERN_MIN_SCORE = 9

MAX_PER_TEMPLATE_IN_KEEP = 6
MAX_PER_TEMPLATE_IN_SHORTLIST = 3
MAX_PER_CATEGORY_IN_SHORTLIST = 20
SHORTLIST_TARGET_SIZE = 180

# Категории, которые иногда слишком head/pop-culture-heavy.
# Не выкидываем, а лишь чуть понижаем вес.
HEAD_HEAVY_CATEGORIES = {"movies", "music", "videogames", "sports", "books"}

print("INPUT_STAGE1_DIR:", INPUT_STAGE1_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


INPUT_STAGE1_DIR: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1
OUTPUT_DIR: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage2


In [4]:

# =========================
# ЗАГРУЗКА
# =========================

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

def detect_input_path() -> Path:
    for path in CANDIDATE_PATHS:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Не найден входной Mintaka-файл. Ожидался один из:\n" +
        "\n".join(str(p) for p in CANDIDATE_PATHS)
    )

INPUT_PATH = detect_input_path()
print("Using input:", INPUT_PATH)

if INPUT_PATH.suffix.lower() == ".jsonl":
    df = read_jsonl(INPUT_PATH)
elif INPUT_PATH.suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_PATH)
else:
    raise ValueError(f"Unsupported input type: {INPUT_PATH.suffix}")

print("raw shape:", df.shape)
print("columns:", list(df.columns))
df.head(3)


Using input: /Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1/jsonl/mintaka_multihop_candidates.jsonl
raw shape: (2000, 27)
columns: ['benchmark_id', 'source_dataset', 'source_split', 'source_id', 'question_en_original', 'question_translations', 'category', 'complexity_type', 'reasoning_family', 'question_entities', 'answer_type', 'answer_mode', 'answer_text_en', 'answer_entities', 'gold_answers', 'gold_qids', 'supporting_entities', 'supporting_numbers', 'num_answers', 'num_question_entities', 'has_supporting_entities', 'has_supporting_numbers', 'needs_translation', 'needs_rephrase', 'translation_source', 'notes', 'priority_score']


,benchmark_id,source_dataset,source_split,source_id,question_en_original,question_translations,category,complexity_type,reasoning_family,question_entities,...,supporting_numbers,num_answers,num_question_entities,has_supporting_entities,has_supporting_numbers,needs_translation,needs_rephrase,translation_source,notes,priority_score
0,mintaka_test_4af81c8c,mintaka,test,4af81c8c,Which awards was the latest Animal Crossing ga...,{'ar': 'ما الجوائز التي ترشحت لها أحدث لعبة من...,videogames,multihop,multihop,"[{'name': 'Q1139803', 'entity_type': 'entity',...",...,[],3,3,False,False,True,True,en,,10
1,mintaka_test_63690199,mintaka,test,63690199,Who produced the movie that's based on the Har...,{'ar': 'من انتج الفيلم المقتبس من كتاب هاري بو...,books,multihop,multihop,"[{'name': 'Q8337', 'entity_type': 'entity', 'l...",...,[],3,2,False,False,True,True,en,,10
2,mintaka_train_11dfec2b,mintaka,train,11dfec2b,Where was the oldest member of The Rolling Sto...,{'ar': 'أين ولد أصغر عضو لفريق (ذا رولينغ ستون...,music,multihop,multihop,"[{'name': 'Q11036', 'entity_type': 'entity', '...",...,[],3,1,False,False,True,True,en,,10


In [5]:

# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================

def safe_str(x: Any) -> str:
    if x is None:
        return ""
    return str(x)

def normalize_spaces(text: Any) -> str:
    text = safe_str(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def to_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # пытаемся распарсить JSON/list literal
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
            try:
                val = json.loads(s)
                if isinstance(val, list):
                    return val
            except Exception:
                pass
        return [s]
    return [x]

def first_nonempty(row: pd.Series, candidates: List[str], default: Any = None) -> Any:
    for c in candidates:
        if c in row and pd.notna(row[c]) and safe_str(row[c]).strip() != "":
            return row[c]
    return default


In [6]:

# =========================
# НОРМАЛИЗАЦИЯ КОЛОНОК
# =========================

def normalize_input(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # benchmark id
    if "benchmark_id" not in out.columns:
        if "id" in out.columns:
            out["benchmark_id"] = out["id"]
        elif "source_id" in out.columns:
            out["benchmark_id"] = out["source_id"]
        else:
            out["benchmark_id"] = [f"mintaka_{i:06d}" for i in range(len(out))]

    # question text
    if "question_en_original" not in out.columns:
        for alt in ["question", "Question", "questionText", "input"]:
            if alt in out.columns:
                out["question_en_original"] = out[alt]
                break

    # category / complexity / family
    if "category" not in out.columns:
        out["category"] = None

    if "complexity_type" not in out.columns:
        for alt in ["complexityType", "complexity", "type"]:
            if alt in out.columns:
                out["complexity_type"] = out[alt]
                break
        else:
            out["complexity_type"] = "unknown"

    if "reasoning_family" not in out.columns:
        if "reasoning_family" not in out.columns:
            out["reasoning_family"] = out["complexity_type"]

    # answer text and num answers
    if "answer_text_en" not in out.columns:
        for alt in ["answer", "answers", "answerText", "answer_text"]:
            if alt in out.columns:
                out["answer_text_en"] = out[alt]
                break

    if "num_answers" not in out.columns:
        nums = []
        for _, row in out.iterrows():
            answers = to_list(first_nonempty(row, ["answer_text_en", "answer", "answers"], default=[]))
            nums.append(len(answers) if answers else (1 if normalize_spaces(first_nonempty(row, ["answer_text_en", "answer", "answers"], default="")) else 0))
        out["num_answers"] = nums

    if "answer_mode" not in out.columns:
        modes = []
        for _, row in out.iterrows():
            q = normalize_spaces(first_nonempty(row, ["question_en_original"], default=""))
            a = first_nonempty(row, ["answer_text_en", "answer", "answers"], default=None)
            ans_list = to_list(a)
            qlow = q.lower()
            if qlow.startswith(("is ", "was ", "were ", "are ", "do ", "does ", "did ", "can ", "has ", "have ")):
                modes.append("boolean")
            elif qlow.startswith("how many"):
                modes.append("count")
            elif len(ans_list) >= 2:
                modes.append("list_like")
            else:
                modes.append("single")
        out["answer_mode"] = modes

    # priority flags from previous stage if present
    if "priority_score" not in out.columns:
        out["priority_score"] = 0

    if "source_dataset" not in out.columns:
        out["source_dataset"] = "mintaka"

    if "source_split" not in out.columns:
        for alt in ["split", "source_split"]:
            if alt in out.columns:
                out["source_split"] = out[alt]
                break
        else:
            out["source_split"] = "unknown"

    if "source_id" not in out.columns:
        out["source_id"] = out["benchmark_id"]

    keep_cols = [
        "benchmark_id",
        "source_dataset",
        "source_split",
        "source_id",
        "question_en_original",
        "category",
        "complexity_type",
        "reasoning_family",
        "answer_mode",
        "answer_text_en",
        "num_answers",
        "priority_score",
    ]
    for c in keep_cols:
        if c not in out.columns:
            out[c] = None

    return out

df = normalize_input(df)
print("normalized shape:", df.shape)
df[["benchmark_id", "question_en_original", "category", "complexity_type", "reasoning_family", "answer_mode", "num_answers"]].head(5)


normalized shape: (2000, 27)


,benchmark_id,question_en_original,category,complexity_type,reasoning_family,answer_mode,num_answers
0,mintaka_test_4af81c8c,Which awards was the latest Animal Crossing ga...,videogames,multihop,multihop,list_like,3
1,mintaka_test_63690199,Who produced the movie that's based on the Har...,books,multihop,multihop,list_like,3
2,mintaka_train_11dfec2b,Where was the oldest member of The Rolling Sto...,music,multihop,multihop,list_like,3
3,mintaka_train_29e5bf22,Who were the playable characters of the fourth...,videogames,multihop,multihop,list_like,3
4,mintaka_train_34409836,Where was the former member of One Direction b...,music,multihop,multihop,list_like,3


In [7]:

# =========================
# ПРИЗНАКИ СЛОЖНОСТИ / ШАБЛОНЫ / SHALLOW-BRIDGE
# =========================

BRIDGE_RELATION_PATTERNS = [
    r"author of",
    r"writer of",
    r"director of",
    r"producer of",
    r"singer of",
    r"lead singer of",
    r"actor of",
    r"actor who played",
    r"actress who played",
    r"quarterback of",
    r"captain of",
    r"drummer of",
    r"wife of",
    r"husband of",
    r"child of",
    r"children of",
    r"member of",
]

SIMPLE_TARGET_PATTERNS = [
    r"born\b",
    r"from\b",
    r"genre\b",
    r"nationality\b",
    r"platform\b",
    r"college\b",
    r"school\b",
    r"hometown\b",
    r"country\b",
    r"city\b",
]

TECHNICAL_PATTERNS = [
    r"zip code",
    r"postal code",
    r"official website",
    r"email address",
    r"coordinate",
    r"iab code",
    r"iso \d+",
    r"rgb color",
    r"hex triplet",
    r"identifier",
    r"exploitation visa",
    r"series ordinal",
    r"mass not equal to",
    r"area greater than",
    r"population (?:greater|less) than",
]

TEMPORAL_PATTERNS = [
    r"\bbefore\b",
    r"\bafter\b",
    r"\bbetween\b",
    r"\bin \d{4}\b",
    r"\bfrom \d{4}\b",
    r"\bsince\b",
    r"\bduring\b",
    r"\bwhen\b",
    r"\bearlier than\b",
    r"\blater than\b",
    r"\boldest\b",
    r"\blatest\b",
    r"\bfirst\b",
    r"\blast\b",
]

COMPARISON_PATTERNS = [
    r"\bmore than\b",
    r"\bless than\b",
    r"\blarger\b",
    r"\bsmaller\b",
    r"\blonger\b",
    r"\bshorter\b",
    r"\bhighest\b",
    r"\blowest\b",
    r"\bbiggest\b",
    r"\blargest\b",
    r"\bsmallest\b",
    r"\btop\b",
]

INTERSECTION_PATTERNS = [
    r"\band\b",
    r"\bwhose\b",
    r"\bthat\b",
    r"\bwhich\b",
    r"\bwith\b",
    r"\balso\b",
    r"\bboth\b",
]

def detect_shallow_bridge(q: str) -> bool:
    ql = q.lower()
    has_bridge = any(re.search(p, ql) for p in BRIDGE_RELATION_PATTERNS)
    has_simple_target = any(re.search(p, ql) for p in SIMPLE_TARGET_PATTERNS)
    wh_simple = ql.startswith(("who ", "where ", "what ", "which "))
    return bool(has_bridge and has_simple_target and wh_simple)

def count_matches(q: str, patterns: List[str]) -> int:
    ql = q.lower()
    return sum(1 for p in patterns if re.search(p, ql))

def build_surface_template(q: str) -> str:
    # довольно грубая абстракция: убираем entity-like spans, числа, даты, url
    s = normalize_spaces(q)
    s = re.sub(r"https?://\S+", "<url>", s)
    s = re.sub(r"'[^']+'", "<quote>", s)
    s = re.sub(r'"[^"]+"', "<quote>", s)
    s = re.sub(r"\b\d{4}-\d{2}-\d{2}\b", "<date>", s)
    s = re.sub(r"\b\d{4}\b", "<year>", s)
    s = re.sub(r"\b\d+(?:\.\d+)?\b", "<num>", s)
    # заменяем последовательности Title Case слов на <ent>
    s = re.sub(r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,5})\b", "<ent>", s)
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_features(row: pd.Series) -> Dict[str, Any]:
    q = normalize_spaces(row["question_en_original"])
    ql = q.lower()
    num_answers = int(row["num_answers"]) if pd.notna(row["num_answers"]) else 0
    answer_mode = normalize_spaces(row["answer_mode"])
    category = normalize_spaces(row["category"]).lower()

    shallow_bridge = detect_shallow_bridge(q)
    technical = any(re.search(p, ql) for p in TECHNICAL_PATTERNS)
    binary = answer_mode == "boolean" or ql.startswith(("is ", "was ", "were ", "are ", "do ", "does ", "did ", "can ", "has ", "have "))
    count_only = answer_mode == "count" or ql.startswith("how many")
    temporal_hits = count_matches(q, TEMPORAL_PATTERNS)
    comparison_hits = count_matches(q, COMPARISON_PATTERNS)
    intersection_hits = count_matches(q, INTERSECTION_PATTERNS)

    # простая оценка "насколько вопрос не плоский"
    score = 0
    score += 5 if normalize_spaces(row["complexity_type"]).lower() == "multihop" else 0
    score += 4 if answer_mode == "list_like" else 0
    score += 2 if num_answers >= 2 else 0
    score += 1 if num_answers >= 3 else 0
    score += min(temporal_hits, 3)
    score += min(comparison_hits, 2) * 2
    score += min(intersection_hits, 3)
    score += 2 if normalize_spaces(row["reasoning_family"]).lower() in {"multihop", "multi_hop", "intersection", "difference", "comparative", "superlative"} else 0

    # лёгкие и технические вещи понижаем
    if shallow_bridge:
        score -= 5
    if binary:
        score -= 6
    if count_only and num_answers <= 1:
        score -= 3
    if technical:
        score -= 2
    if category in HEAD_HEAVY_CATEGORIES:
        score -= 1

    # дополнительных условий мало -> слабее
    extra_constraints = temporal_hits + comparison_hits + max(0, intersection_hits - 1)
    if shallow_bridge and extra_constraints == 0 and num_answers <= 2:
        score -= 3

    return {
        "surface_template": build_surface_template(q),
        "is_shallow_bridge": shallow_bridge,
        "is_technical": technical,
        "is_binary_question": binary,
        "is_count_only": count_only,
        "temporal_hits": temporal_hits,
        "comparison_hits": comparison_hits,
        "intersection_hits": intersection_hits,
        "extra_constraints": extra_constraints,
        "reconstructability_score_v2": int(score),
    }

features_df = pd.DataFrame([extract_features(r) for _, r in df.iterrows()])
df2 = pd.concat([df.reset_index(drop=True), features_df], axis=1)

print("df2 shape:", df2.shape)
df2[[
    "question_en_original",
    "category",
    "answer_mode",
    "num_answers",
    "is_shallow_bridge",
    "temporal_hits",
    "comparison_hits",
    "intersection_hits",
    "reconstructability_score_v2",
]].head(10)


df2 shape: (2000, 37)


,question_en_original,category,answer_mode,num_answers,is_shallow_bridge,temporal_hits,comparison_hits,intersection_hits,reconstructability_score_v2
0,Which awards was the latest Animal Crossing ga...,videogames,list_like,3,False,1,0,1,15
1,Who produced the movie that's based on the Har...,books,list_like,3,False,1,0,1,15
2,Where was the oldest member of The Rolling Sto...,music,list_like,3,True,1,0,0,9
3,Who were the playable characters of the fourth...,videogames,list_like,3,False,0,0,1,14
4,Where was the former member of One Direction b...,music,list_like,3,True,0,0,0,8
5,Where was the singer of Back in Black born?,music,list_like,3,True,0,0,0,8
6,Who are the love interests in Mass Effect 1?,videogames,list_like,3,False,0,0,0,13
7,Where was Los Angeles Angels' two-way star born?,sports,list_like,3,False,0,0,0,13
8,Where is the lead singer of The Backstreet Boy...,music,list_like,2,True,0,0,0,4
9,Where was the captain of the US women's nation...,sports,list_like,2,True,1,0,1,9


In [8]:

# =========================
# РЕШЕНИЕ: keep / pattern_only / drop
# =========================

def decision_rules(row: pd.Series) -> Tuple[str, str]:
    reasons = []

    if row["is_binary_question"]:
        reasons.append("binary_question")
    if row["is_technical"]:
        reasons.append("technical_fact")
    if row["is_count_only"] and int(row["num_answers"]) <= 1:
        reasons.append("count_only_simple")
    if row["is_shallow_bridge"]:
        reasons.append("shallow_bridge")
    if int(row["num_answers"]) >= 2:
        reasons.append("multi_answer")
    if row["answer_mode"] == "list_like":
        reasons.append("list_like")
    if int(row["temporal_hits"]) > 0:
        reasons.append("temporal_signal")
    if int(row["comparison_hits"]) > 0:
        reasons.append("comparison_signal")
    if int(row["intersection_hits"]) >= 2:
        reasons.append("multi_constraint_signal")

    score = int(row["reconstructability_score_v2"])

    # Жёсткий дроп
    if row["is_binary_question"]:
        return "drop", "|".join(reasons)

    if row["is_shallow_bridge"] and score < PATTERN_MIN_SCORE:
        return "drop", "|".join(reasons)

    # pattern_only
    if row["is_shallow_bridge"] and score >= PATTERN_MIN_SCORE:
        return "pattern_only", "|".join(reasons)

    if row["is_count_only"] and row["answer_mode"] != "list_like":
        return "pattern_only", "|".join(reasons)

    # keep
    if score >= KEEP_MIN_SCORE:
        return "keep_for_reconstruction", "|".join(reasons)

    if score >= PATTERN_MIN_SCORE:
        return "pattern_only", "|".join(reasons)

    return "drop", "|".join(reasons)

decisions = [decision_rules(r) for _, r in df2.iterrows()]
df2["auto_decision"] = [d[0] for d in decisions]
df2["decision_reasons"] = [d[1] for d in decisions]

df2["auto_decision"].value_counts(dropna=False)


auto_decision
drop                       1690
pattern_only                282
keep_for_reconstruction      28
Name: count, dtype: int64

In [9]:

# =========================
# ДЕДУП ПО ШАБЛОНАМ И БАЛАНС
# =========================

def per_template_rank(df_sub: pd.DataFrame) -> pd.DataFrame:
    return (
        df_sub.sort_values(
            by=["reconstructability_score_v2", "num_answers", "priority_score", "benchmark_id"],
            ascending=[False, False, False, True],
        )
        .groupby("surface_template", as_index=False, group_keys=False)
        .head(MAX_PER_TEMPLATE_IN_KEEP)
        .reset_index(drop=True)
    )

keep_df = df2[df2["auto_decision"] == "keep_for_reconstruction"].copy()
pattern_df = df2[df2["auto_decision"] == "pattern_only"].copy()
drop_df = df2[df2["auto_decision"] == "drop"].copy()

keep_df = per_template_rank(keep_df)
pattern_df = (
    pattern_df.sort_values(
        by=["reconstructability_score_v2", "num_answers", "priority_score", "benchmark_id"],
        ascending=[False, False, False, True],
    )
    .groupby("surface_template", as_index=False, group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

print("keep_df:", keep_df.shape)
print("pattern_df:", pattern_df.shape)
print("drop_df:", drop_df.shape)


keep_df: (28, 39)
pattern_df: (267, 39)
drop_df: (1690, 39)


In [10]:

# =========================
# PATTERN BANK
# =========================

pattern_bank = (
    keep_df.sort_values(
        by=["reconstructability_score_v2", "num_answers", "benchmark_id"],
        ascending=[False, False, True]
    )
    .groupby(["surface_template", "category", "answer_mode"], as_index=False)
    .agg(
        n_examples=("benchmark_id", "count"),
        top_score=("reconstructability_score_v2", "max"),
        top_num_answers=("num_answers", "max"),
        example_benchmark_id=("benchmark_id", "first"),
        example_question=("question_en_original", "first"),
        example_answer=("answer_text_en", "first"),
    )
    .sort_values(by=["n_examples", "top_score"], ascending=[False, False])
    .reset_index(drop=True)
)

print("pattern_bank:", pattern_bank.shape)
pattern_bank.head(10)


pattern_bank: (28, 9)


,surface_template,category,answer_mode,n_examples,top_score,top_num_answers,example_benchmark_id,example_question,example_answer
0,<ent> awards was the latest <ent> game nominat...,videogames,list_like,1,15,3,mintaka_test_4af81c8c,Which awards was the latest Animal Crossing ga...,"Best Family Game, Best Multiplayer Game, Game ..."
1,<ent> produced the movie that's based on the <...,books,list_like,1,15,3,mintaka_test_63690199,Who produced the movie that's based on the Har...,"Chris Columbus, David Heyman, Mark Radcliffe"
2,<ent> were the first two senators to represent...,politics,list_like,1,15,2,mintaka_test_5bd63af8,Who were the first two senators to represent t...,Hiram Fong and Oren E. Long
3,<ent> wives of us presidents were also mothers...,history,list_like,1,15,2,mintaka_train_08cbd2e7,Which wives of US presidents were also mothers...,"Abagail Adams, Barbara Bush"
4,<ent> city did the band that wrote <quote> com...,music,list_like,1,14,2,mintaka_train_b9354cf6,"Which city did the band that wrote ""Shiny Happ...","Athens, GA"
5,<ent> was the first president of the <ent> born?,history,list_like,1,14,2,mintaka_train_05e10f57,Where was the first president of the United St...,"Westmoreland County, Virginia"
6,<ent> was the top scorer of the <ent> tourname...,sports,list_like,1,14,2,mintaka_validation_304b59eb,Who was the top scorer of the Copa America tou...,"Lionel Messi, Luis Diaz"
7,<ent> were the playable characters of the four...,videogames,list_like,1,14,3,mintaka_train_29e5bf22,Who were the playable characters of the fourth...,"Niko Bellic, Johnny Klebitz, Luis Fernando Lopez"
8,<ent> are the love interests in <ent> <num>?,videogames,list_like,1,13,3,mintaka_train_91c7335e,Who are the love interests in Mass Effect 1?,"Kaidan Alenko, Ashley Williams, Liara T'Soni"
9,<ent> characters were <ent>'s best friends and...,videogames,list_like,1,13,2,mintaka_train_ec2f1490,What Disney characters were Sora's best friend...,Donald Duck and Goofy


In [11]:

# =========================
# SHORTLIST ДЛЯ РУЧНОГО ПРОСМОТРА
# =========================

shortlist_base = keep_df.copy()

# Сначала ограничиваем по шаблону
shortlist_base = (
    shortlist_base.sort_values(
        by=["reconstructability_score_v2", "num_answers", "priority_score", "benchmark_id"],
        ascending=[False, False, False, True],
    )
    .groupby("surface_template", as_index=False, group_keys=False)
    .head(MAX_PER_TEMPLATE_IN_SHORTLIST)
    .reset_index(drop=True)
)

# Затем балансируем по категориям, чтобы не завалиться в один домен
shortlist_base = (
    shortlist_base.sort_values(
        by=["reconstructability_score_v2", "num_answers", "priority_score", "benchmark_id"],
        ascending=[False, False, False, True],
    )
    .groupby("category", as_index=False, group_keys=False)
    .head(MAX_PER_CATEGORY_IN_SHORTLIST)
    .reset_index(drop=True)
)

# Финальное ограничение размера
reconstruction_shortlist = (
    shortlist_base.sort_values(
        by=["reconstructability_score_v2", "num_answers", "priority_score", "benchmark_id"],
        ascending=[False, False, False, True],
    )
    .head(SHORTLIST_TARGET_SIZE)
    .reset_index(drop=True)
)

print("reconstruction_shortlist:", reconstruction_shortlist.shape)
reconstruction_shortlist[[
    "benchmark_id",
    "category",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "reconstructability_score_v2",
    "question_en_original",
]].head(20)


reconstruction_shortlist: (28, 39)


,benchmark_id,category,reasoning_family,answer_mode,num_answers,reconstructability_score_v2,question_en_original
0,mintaka_test_4af81c8c,videogames,multihop,list_like,3,15,Which awards was the latest Animal Crossing ga...
1,mintaka_test_63690199,books,multihop,list_like,3,15,Who produced the movie that's based on the Har...
2,mintaka_test_5bd63af8,politics,multihop,list_like,2,15,Who were the first two senators to represent t...
3,mintaka_train_08cbd2e7,history,multihop,list_like,2,15,Which wives of US presidents were also mothers...
4,mintaka_train_29e5bf22,videogames,multihop,list_like,3,14,Who were the playable characters of the fourth...
5,mintaka_train_05e10f57,history,multihop,list_like,2,14,Where was the first president of the United St...
6,mintaka_train_b9354cf6,music,multihop,list_like,2,14,"Which city did the band that wrote ""Shiny Happ..."
7,mintaka_validation_304b59eb,sports,multihop,list_like,2,14,Who was the top scorer of the Copa America tou...
8,mintaka_train_91c7335e,videogames,multihop,list_like,3,13,Who are the love interests in Mass Effect 1?
9,mintaka_train_b5103288,sports,multihop,list_like,3,13,Where was Los Angeles Angels' two-way star born?


In [12]:

# =========================
# MANUAL REVIEW SHEET
# =========================

manual_review_sheet = reconstruction_shortlist[[
    "benchmark_id",
    "category",
    "complexity_type",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "reconstructability_score_v2",
    "surface_template",
    "question_en_original",
    "answer_text_en",
]].copy()

manual_review_sheet["manual_decision"] = ""
manual_review_sheet["manual_bucket"] = ""
manual_review_sheet["mintaka_reconstruction_idea"] = ""
manual_review_sheet["target_domain_ru"] = ""
manual_review_sheet["notes"] = ""

manual_review_sheet.head(10)


,benchmark_id,category,complexity_type,reasoning_family,answer_mode,num_answers,reconstructability_score_v2,surface_template,question_en_original,answer_text_en,manual_decision,manual_bucket,mintaka_reconstruction_idea,target_domain_ru,notes
0,mintaka_test_4af81c8c,videogames,multihop,multihop,list_like,3,15,<ent> awards was the latest <ent> game nominat...,Which awards was the latest Animal Crossing ga...,"Best Family Game, Best Multiplayer Game, Game ...",,,,,
1,mintaka_test_63690199,books,multihop,multihop,list_like,3,15,<ent> produced the movie that's based on the <...,Who produced the movie that's based on the Har...,"Chris Columbus, David Heyman, Mark Radcliffe",,,,,
2,mintaka_test_5bd63af8,politics,multihop,multihop,list_like,2,15,<ent> were the first two senators to represent...,Who were the first two senators to represent t...,Hiram Fong and Oren E. Long,,,,,
3,mintaka_train_08cbd2e7,history,multihop,multihop,list_like,2,15,<ent> wives of us presidents were also mothers...,Which wives of US presidents were also mothers...,"Abagail Adams, Barbara Bush",,,,,
4,mintaka_train_29e5bf22,videogames,multihop,multihop,list_like,3,14,<ent> were the playable characters of the four...,Who were the playable characters of the fourth...,"Niko Bellic, Johnny Klebitz, Luis Fernando Lopez",,,,,
5,mintaka_train_05e10f57,history,multihop,multihop,list_like,2,14,<ent> was the first president of the <ent> born?,Where was the first president of the United St...,"Westmoreland County, Virginia",,,,,
6,mintaka_train_b9354cf6,music,multihop,multihop,list_like,2,14,<ent> city did the band that wrote <quote> com...,"Which city did the band that wrote ""Shiny Happ...","Athens, GA",,,,,
7,mintaka_validation_304b59eb,sports,multihop,multihop,list_like,2,14,<ent> was the top scorer of the <ent> tourname...,Who was the top scorer of the Copa America tou...,"Lionel Messi, Luis Diaz",,,,,
8,mintaka_train_91c7335e,videogames,multihop,multihop,list_like,3,13,<ent> are the love interests in <ent> <num>?,Who are the love interests in Mass Effect 1?,"Kaidan Alenko, Ashley Williams, Liara T'Soni",,,,,
9,mintaka_train_b5103288,sports,multihop,multihop,list_like,3,13,<ent> was <ent>' two-way star born?,Where was Los Angeles Angels' two-way star born?,"Oshu, Iwate, Japan",,,,,


In [13]:

# =========================
# СОХРАНЕНИЕ
# =========================

def df_to_jsonl(df_: pd.DataFrame, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for rec in df_.to_dict(orient="records"):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def df_to_json(df_: pd.DataFrame, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(df_.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

def save_triplet(df_: pd.DataFrame, stem: str) -> None:
    (OUT_CSV_DIR / f"{stem}.csv").parent.mkdir(parents=True, exist_ok=True)
    df_.to_csv(OUT_CSV_DIR / f"{stem}.csv", index=False)
    df_to_jsonl(df_, OUT_JSONL_DIR / f"{stem}.jsonl")
    df_to_json(df_, OUT_JSONL_DIR / f"{stem}.json")

save_triplet(df2, "mintaka_scored_all")
save_triplet(keep_df, "mintaka_filtered_candidates")
save_triplet(pattern_df, "mintaka_pattern_only")
save_triplet(pattern_bank, "mintaka_pattern_bank")
save_triplet(reconstruction_shortlist, "mintaka_reconstruction_shortlist")
save_triplet(manual_review_sheet, "mintaka_manual_review_sheet")

summary = {
    "input_path": str(INPUT_PATH),
    "raw_rows": int(len(df)),
    "scored_rows": int(len(df2)),
    "decision_counts": {k: int(v) for k, v in df2["auto_decision"].value_counts().to_dict().items()},
    "keep_rows_after_template_filter": int(len(keep_df)),
    "pattern_rows_after_template_filter": int(len(pattern_df)),
    "pattern_bank_rows": int(len(pattern_bank)),
    "shortlist_rows": int(len(reconstruction_shortlist)),
    "answer_mode_counts_shortlist": {k: int(v) for k, v in reconstruction_shortlist["answer_mode"].value_counts().to_dict().items()},
    "category_counts_shortlist": {k: int(v) for k, v in reconstruction_shortlist["category"].value_counts().to_dict().items()},
}

with open(OUT_REPORTS_DIR / "mintaka_stage2_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary


{'input_path': '/Users/matvey/Desktop/mintaka/artifacts_mintaka_stage1/jsonl/mintaka_multihop_candidates.jsonl',
 'raw_rows': 2000,
 'scored_rows': 2000,
 'decision_counts': {'drop': 1690,
  'pattern_only': 282,
  'keep_for_reconstruction': 28},
 'keep_rows_after_template_filter': 28,
 'pattern_rows_after_template_filter': 267,
 'pattern_bank_rows': 28,
 'shortlist_rows': 28,
 'answer_mode_counts_shortlist': {'list_like': 26, 'single': 2},
 'category_counts_shortlist': {'videogames': 7,
  'books': 5,
  'sports': 5,
  'politics': 4,
  'music': 3,
  'history': 2,
  'geography': 2}}

In [14]:

# =========================
# БЫСТРЫЙ ПРЕВЬЮ
# =========================

print("=== decision counts ===")
print(df2["auto_decision"].value_counts(dropna=False))
print()

print("=== shortlist by category ===")
print(reconstruction_shortlist["category"].value_counts().head(15))
print()

print("=== shortlist by answer_mode ===")
print(reconstruction_shortlist["answer_mode"].value_counts(dropna=False))
print()

reconstruction_shortlist[[
    "benchmark_id",
    "category",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "reconstructability_score_v2",
    "question_en_original",
    "answer_text_en",
]].head(30)


=== decision counts ===
auto_decision
drop                       1690
pattern_only                282
keep_for_reconstruction      28
Name: count, dtype: int64

=== shortlist by category ===
category
videogames    7
books         5
sports        5
politics      4
music         3
history       2
geography     2
Name: count, dtype: int64

=== shortlist by answer_mode ===
answer_mode
list_like    26
single        2
Name: count, dtype: int64



,benchmark_id,category,reasoning_family,answer_mode,num_answers,reconstructability_score_v2,question_en_original,answer_text_en
0,mintaka_test_4af81c8c,videogames,multihop,list_like,3,15,Which awards was the latest Animal Crossing ga...,"Best Family Game, Best Multiplayer Game, Game ..."
1,mintaka_test_63690199,books,multihop,list_like,3,15,Who produced the movie that's based on the Har...,"Chris Columbus, David Heyman, Mark Radcliffe"
2,mintaka_test_5bd63af8,politics,multihop,list_like,2,15,Who were the first two senators to represent t...,Hiram Fong and Oren E. Long
3,mintaka_train_08cbd2e7,history,multihop,list_like,2,15,Which wives of US presidents were also mothers...,"Abagail Adams, Barbara Bush"
4,mintaka_train_29e5bf22,videogames,multihop,list_like,3,14,Who were the playable characters of the fourth...,"Niko Bellic, Johnny Klebitz, Luis Fernando Lopez"
5,mintaka_train_05e10f57,history,multihop,list_like,2,14,Where was the first president of the United St...,"Westmoreland County, Virginia"
6,mintaka_train_b9354cf6,music,multihop,list_like,2,14,"Which city did the band that wrote ""Shiny Happ...","Athens, GA"
7,mintaka_validation_304b59eb,sports,multihop,list_like,2,14,Who was the top scorer of the Copa America tou...,"Lionel Messi, Luis Diaz"
8,mintaka_train_91c7335e,videogames,multihop,list_like,3,13,Who are the love interests in Mass Effect 1?,"Kaidan Alenko, Ashley Williams, Liara T'Soni"
9,mintaka_train_b5103288,sports,multihop,list_like,3,13,Where was Los Angeles Angels' two-way star born?,"Oshu, Iwate, Japan"



## Что смотреть после запуска

Главные файлы:

- `artifacts_mintaka_stage2/jsonl/mintaka_filtered_candidates.jsonl`
- `artifacts_mintaka_stage2/jsonl/mintaka_pattern_bank.jsonl`
- `artifacts_mintaka_stage2/jsonl/mintaka_reconstruction_shortlist.jsonl`
- `artifacts_mintaka_stage2/csv/mintaka_manual_review_sheet.csv`

Логика чтения:
- `filtered_candidates` — всё, что осталось после автоматического отсечения;
- `pattern_bank` — сгруппированные шаблоны, чтобы не тонуть в повторах;
- `reconstruction_shortlist` — компактный shortlist для просмотра руками;
- `manual_review_sheet` — таблица, где удобно ставить `keep / pattern_only / drop`.
